# Processing the data

This is how to train a sequence classifier on one batch.

In [1]:
import torch
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Same as before
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
sequences = [
    "I've been waiting for a HuggingFace course my whole life.",
    "This course is amazing!",
]
batch = tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")

# This is new
batch["labels"] = torch.tensor([1, 1])

optimizer = AdamW(model.parameters())
loss = model(**batch).loss
loss.backward()
optimizer.step()

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Loading a dataset from the Hub

The HF datasets library provides a very simple command to download and cache a dataset on the Hub.

In [4]:
!pip install datasets

In [6]:
# Downloadin the MRPC dataset
from datasets import load_dataset

raw_dataset = load_dataset("SetFit/mrpc")
raw_dataset

README.md:   0%|          | 0.00/316 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

validation.jsonl:   0%|          | 0.00/127k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/533k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text1', 'text2', 'label', 'idx', 'label_text'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['text1', 'text2', 'label', 'idx', 'label_text'],
        num_rows: 408
    })
    test: Dataset({
        features: ['text1', 'text2', 'label', 'idx', 'label_text'],
        num_rows: 1725
    })
})

We get a `DatasetDict` which contains the training set, the validation set and the test set.

We can access each pair of sentences in our `raw_dataset` object by indexing, like a dictonary.

In [8]:
raw_train_dataset = raw_dataset['train']
raw_train_dataset[0]

{'text1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'text2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0,
 'label_text': 'equivalent'}

To know which integer corresponds to which label, we can inspect the `features` of our raw_train_dataset.

In [9]:
raw_train_dataset.features

{'text1': Value('string'),
 'text2': Value('string'),
 'label': Value('int64'),
 'idx': Value('int64'),
 'label_text': Value('string')}

## Processing a dataset

To process the dataset, we need to convert the text to numbers the model can make sense of.

In [10]:
from transformers import AutoTokenizer

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
tokenized_text_1 = tokenizer.tokenize(raw_train_dataset[0]['text1'])
tokenized_text_2 = tokenizer.tokenize(raw_train_dataset[0]['text2'])

We can use a pair of sequence also

In [11]:
inputs = tokenizer("This is the first sentence.", "This is the second one.")
inputs

{'input_ids': [101, 2023, 2003, 1996, 2034, 6251, 1012, 102, 2023, 2003, 1996, 2117, 2028, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

`token_type_ids` -> tells the model which part of input is the first sentence and which is second sentence.

BERT is pretrained with token type IDs, and on top of the masked language modeling objective, it has an additional objective called `next sentence prediction`. The goal with this task is to model the relationship between pairs of sentences.

**If we change the checkpoint we won't necessarily get the `token_type_ids` in our tokenized inputs.**

Tokenizing the whole dataset

In [20]:
tokenized_dataset = tokenizer(
    list(raw_dataset['train']['text1']),
    list(raw_dataset['train']['text2']),
    padding = True,
    truncation = True
    )

This has the following disadvantage:
* Returning a dict
* It will only work if we have enough RAM to store oue entire dataset during tokenization

To keep the data as dataset, we use the `Dataset.map()` method.

The `map()` method works by applying a function on each element of the dataset

In [24]:
def tokenize_function(example):
    return tokenizer(example["text1"], example["text2"], truncation=True)

We apply the tokenization function on all our datasets at once. We’re using `batched=True` in our call to map so the function is applied to multiple elements of our dataset at once, and not on each element separately. This allows for faster preprocessing.

In [25]:
tokenized_datasets = raw_dataset.map(tokenize_function, batched=True)
tokenized_datasets

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text1', 'text2', 'label', 'idx', 'label_text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['text1', 'text2', 'label', 'idx', 'label_text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['text1', 'text2', 'label', 'idx', 'label_text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

The way the HF Datasets library applies this processing is by adding new fields to the datasets, one for each key in the dictionary returned by the preprocessing function

## Dynamic Padding

The function that is responsible for putting together samples inside a batch is called a `collate function.` It’s an argument you can pass when you build a DataLoader, the default being a function that will just convert your samples to PyTorch tensors and concatenate them (recursively if your elements are lists, tuples, or dictionaries).

his won’t be possible in our case since the inputs we have won’t all be of the same size. We have deliberately postponed the padding, to only apply it as necessary on each batch and avoid having over-long inputs with a lot of padding.

To do this in practice, we have to define a collate function that will apply the correct amount of padding to the items of the dataset we want to batch together.

In [26]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [29]:
samples = tokenized_datasets["train"][:8]
samples = {k: v for k, v in samples.items() if k not in ["idx", "text1", "text2", "label_text"]}
[len(x) for x in samples["input_ids"]]

[50, 59, 47, 67, 59, 50, 62, 32]

Dynamic padding means the samples in this batch should all be padded to a length of 67, the maximum length inside the batch. Without dynamic padding, all of the samples would have to be padded to the maximum length in the whole dataset, or the maximum length the model can accept.

In [30]:
batch = data_collator(samples)
{k: v.shape for k, v in batch.items()}

{'input_ids': torch.Size([8, 67]),
 'token_type_ids': torch.Size([8, 67]),
 'attention_mask': torch.Size([8, 67]),
 'labels': torch.Size([8])}